# 2.3 Analyzing the model

## Live code in this page

This page uses **live code**. Click the rocket button (🚀) at the top of this page, above the title, then choose "▶ Live Code". The loading may take some time so it is good to start the Live Code before reading on. The Live Code engine is ready when you see a text under the page's title, stating something like "Launching from mybinder.org: **ready**.

**Note:** Code that you edit in the live code blocks is not saved. Any changes you make to the code blocks is lost once you move on from the page. Therefore, if you want to keep your work, save a copy of it in a local document.

The code block below defines again the `crop_definition` dictionary that we've worked with in previous sections. It also defines again the `display_dict` function which we've used to display dictionaries in a convenient way. Since you have seen this code before, there is no need to go over it. It is run automatically when you click the "▶ Live Code" button.

In [ ]:
"""
Create a function for displaying dicts in an aesthetic way.
Consider the option of a nested dict (like a GreenLight model definition)
or a "flat" dict (just keys and values)
"""

import pandas as pd

def display_dict(d, dict_type="nested"):
    """
    Below, some dictionaries will be defined in the standard
    GreenLight format for describing models.
    This functions displays such dictionaries in a nicely
    readable format
    """
    if dict_type == "nested":
        df = pd.DataFrame.from_dict(d, orient="index")
        df.index.name = "Variable"
    elif dict_type == "flat":
        df = pd.DataFrame.from_dict(d, orient="index", columns=[""])
    display(df)


"""
Solution to Exercise 6: complete the crop definition dict with description, unit, type, definition, and reference
"""

crop_definition = {
    "W": {
        "description": "Crop dry mass",
        "unit": "g/m2",
        "type": "state",
        "definition": "LUE * ( 1 - exp(-k*LAI) ) * (1/1000000 * Iw)",  # The expression for dW/dT (for states, the definition is the right hand sand of the ODE)
        "init": "75",
        "reference": "Stanghellini (2019), Eq. 2.25. I (MJ/m2/day) is replaced by Iw (W/m2). See also Figure 2.13 there: 75 g/m2 dry mass is a reasonable value for a small crop"
    },
    "LUE": {
        "description": "Light use efficiency",
        "unit": "g crop weight per MJ PAR",
        "type": "const",
        "definition": "4.3",
        "reference": "Stanghellini (2019), Eq. 2.25. See also pg. 49 there: 4.3 is a typical middle value"
    },
    "k": {
        "description": "Light extinction coefficient",
        "unit": "-",
        "type": "const",
        "definition": "0.7",
        "reference": "Stanghellini (2019), Eq. 2.25. See also pg. 39 there: 'For many greenhouse crops, k has a value between 0.6 and 0.8'"
    },
    "LAI": {
        "description": "Leaf area index",
        "unit": "m2 leaf per m2 floor",
        "type": "aux",
        "definition": "W * F_leaves * SLA / 10000",
        "reference": "Stanghellini (2019), Eq. 2.26. "
    },
    "Iw": {
        "description": "Solar PAR radiation",
        "unit": "W PAR per m2",
        "type": "input",
        "definition": "50",
        "reference": "Stanghellini (2019), Eq. 2.25. I (MJ/m2/day) has been replaced by Iw (W/m2). The value of 50 W/m2 is given as an example"
    },
    "F_leaves": {
        "description": "Fraction of crop dry mass partitioned to the leaves",
        "unit": "-",
        "type": "const",
        "definition": "0.8",
        "reference": "Stanghellini (2019), Eq. 2.26. 0.8 is chosen to represent a vegetative crop with mostly leaves"
    },
    "SLA": {
        "description": "Specific leaf area",
        "unit": "cm2/g",
        "type": "const",
        "definition": "250",
        "reference": "Stanghellini (2019), Eq. 2.26. 250 is a middle value for a tomato crop (see Table 2.1, pg. 50 in Stanghellini 2019)"
    }
 }

# Display the above dict
display_dict(crop_definition)

## Creating an interactive model tool

In this page we will now create an interactive [widget](https://ipywidgets.readthedocs.io/en/latest/) which will allow to modify, run, and view model results in one set of graphs. This page includes some detailed Python coding, but note that this is not the main focus of the lesson. The goal is to use the widget in order to analyze and understand the crop model.

The following code blocks set up the widget. Go through them one by one and run the code.

The first code block sets up which model components can be adjusted by the interactive widget, and which model variables are displayed as outputs. You may later try to modify this block and see what happens.

In [ ]:
"""
Set up the control sliders and output variables of the interactive widget
"""

# Control sliders: variables that can be modified before running the model
# ------------------------------------------------------------------------

# The names that will appear next to each slider
slider_names = ["W init", "Iw", "LUE"]

# The default start value for each slider (ordered as above)
start_values = [75,       50,   4.3  ]

# The minimum allowed values
min_values   = [0,        0,    0    ]

# The maximum allowed values
max_values   = [250,      250,  20   ]

# Step size: how accurate the values are allowed to be
step_sizes   = [0.1,      1,    0.01 ]

# For each slider, the variable it controls in crop_definition
var_names    = ["W",    "Iw",         "LUE"]

# The field of the variable that is modified
#   (for consts, this is the "definition",
#   but for initial values, this is "init")
var_fields   = ["init", "definition", "definition"]


# Output variables: the model outputs that will be displayed
# ----------------------------------------------------------
output_vars = ["W", "LAI"]

# Show setup
# -------------
print("Sliders setup:")
display(pd.DataFrame(
  [slider_names, start_values, min_values, max_values, step_sizes, var_names, var_fields],
  index=["Slider","Start","Min","Max","Step","Variable","Field"]
))

print("Output variables:")
print(output_vars)

The following code block defines the widget, buttons, and the functions it runs. For this course, it is not necessary to go through this code. You may simply run it.

In [ ]:
"""
Define the widget, buttons, and the functions it runs
"""

import greenlight
import traceback
import ipywidgets as widgets
import matplotlib.pyplot as plt
import copy

from IPython.display import display

%matplotlib widget

# Output widget to display text output, including errors if they arise
output = widgets.Output()

# Define the input sliders
sliders = []
for name, start_val, min_val, max_val, step in zip(slider_names, start_values, min_values, max_values, step_sizes):
    sliders.append(
        widgets.FloatSlider(
            value=start_val,
            min=min_val,
            max=max_val,
            step=step,
            description=name,
            continuous_update=False
        )
    )


# Define the buttons
run_button = widgets.Button(
    description='Run model',
    button_style='success',
    icon='play'
)
delete_button = widgets.Button(
    description='Delete latest curve',
    button_style='danger',
    icon='trash'
)

# Function to run the model
def run_model(button):

    with output:
        output.clear_output(wait=True)

        try:
            # For debugging purposes: show that the model has started
            print("Running model...")

            # Read slider values
            s_values = [slider.value for slider in sliders]

            # Create a new copy of the model definition dict
            crop_definition_new = copy.deepcopy(crop_definition)

            # Modify the model definition according to the sliders
            for val, var, field in zip(s_values, var_names, var_fields):
                crop_definition_new[var][field] = str(val)

            # Create and run the model
            model = greenlight.GreenLight(input_prompt=crop_definition_new)
            model.run()

            # Load model results
            data = model.full_sol

            # For each generated curve, the label describes the chosen
            # slider values
            label = ", ".join(
                f"{name}: {value}"
                for name, value in zip(slider_names, s_values)
            )

            for ax, ax_lines, o_var in zip(axes, axes_lines, output_vars):
              line, = ax.plot(
                data["Time"] / 3600,  # Display the time in hours instead of seconds
              	data[o_var],
              	label=label
              )

              ax_lines.append(line)
              ax.legend()
              ax.relim()
              ax.autoscale(enable=None, axis="y", tight=True)

              fig.canvas.draw_idle()

            # For debugging purposes
            print("Model successfully plotted!")

        except Exception:
            traceback.print_exc()


# Function to delete the latest curve
def delete_latest(button):

    if axes_lines:
      for ax, ax_lines in zip(axes, axes_lines):

        # Remove the most recent curve
        line = ax_lines.pop()
        line.remove()

        # Update legend and rescale figure
        ax.legend(loc='best')
        ax.relim()
        ax.autoscale(enable=None, axis="y", tight=True)

        fig.canvas.draw_idle()



# Connect buttons to functions
run_button.on_click(run_model)
delete_button.on_click(delete_latest)

# Show buttons generated
print("Buttons generated:")
print(run_button)
print(delete_button)

Finally, this last code block sets up the interface and runs the widget. You may run it and start interacting with the model.

In [ ]:
"""
Set up the widget and run it
"""

# Number of subplots is equal to the number of output variables
n_subplots = len(output_vars)

# Create the figure with subplots
fig, axes = plt.subplots(
  n_subplots, 1,
  figsize=(8, 3 * n_subplots),
  sharex=True,
  squeeze=False
)
axes = axes.flatten()
fig.subplots_adjust(hspace=0.5)

# A list containing the lines created
axes_lines = []

# For each output variable, set up the generated figure
for o_var, ax in zip(output_vars, axes):
  ax.set_xlabel('Time (hours)')
  ax.set_ylabel(o_var + " (" + crop_definition[o_var]["unit"] + ")")
  ax.set_title(crop_definition[o_var]["description"])
  ax.grid(True, alpha=0.3)
  ax.set_xlim(0, 23)

  axes_lines.append([])

# Set up the control buttons
controls = widgets.VBox([
    widgets.HTML('<h3>Model parameters</h3>'),
    *sliders,
    widgets.HBox([
        run_button,
        delete_button
    ])
])

# Display the widget and allow the user to interact with it
display(controls)

# For debugging purposes: display output text (including errors)
# display(output)